# RuBERT-tiny2: обучение головы выбора

Проверяем русский трёхслойный encoder с новой скалярной головой: запрос → E5 top-20 → выбранный ID. Готового reranker здесь нет: голова создана с seed 20261007, исходные encoder-веса сохранены точно.

Цель: измерить качество на сложном STT и локальную скорость после обучения. Laya остаётся основной моделью приложения. Предположение о скорости проверяется на CPU.

## Протокол до просмотра результатов

Те же 22 656 train: 5 664 clean и 16 992 STT-искажения. Для выбора весов — прежние 150 synthetic validation, разделённые по группам.

Первые 2 272 train-строки чистые, затем смесь остальных; максимум один проход. FP32, AdamW lr=2e-5, effective batch=16, micro=2, warmup=6%, weight decay=.01. Validation NLL каждые 142 обновления, patience=3, min_delta=.005. Исходные веса с новой головой также участвуют в выборе.

Личные каталоги и test не загружаются. Известные 800 представлений будут проверены локально после выбора checkpoint; они не будут новым слепым тестом.

In [ ]:
import os, sys, json, hashlib, subprocess
from pathlib import Path
os.environ.update(USE_TF='0', USE_FLAX='0', TOKENIZERS_PARALLELISM='false')
subprocess.check_call([sys.executable, '-m', 'pip', 'install', '--quiet', 'transformers==5.12.1', 'safetensors==0.6.2'])
import torch
assert torch.cuda.is_available(), 'Нужен GPU Kaggle'
print({'torch':torch.__version__, 'gpu':torch.cuda.get_device_name(0)})


## Пары и динамический выбор

BERT получает запрос и одну карточку с token_type_ids 0/1. Голова выдаёт одно число для каждой пары. Двадцать чисел сравниваются через cross-entropy. Target ID нужен только для индекса правильного варианта, без класса под каждое приложение. Порядок кандидатов закреплён исходными данными.

Валидационные промахи E5 сохраняются в accuracy цепочки. Условный NLL для выбора весов считается только там, где цель есть среди кандидатов.

In [ ]:
matches = list(Path('/kaggle/input').rglob('rubert-input-manifest.json'))
assert len(matches)==1, f'Ожидался один пакет, найдено {len(matches)}'
input_dir = matches[0].parent
assert hashlib.sha256(matches[0].read_bytes()).hexdigest() == 'a660d8648538feb98047dc5a716375e4c3a744e8e7ecd72b38144a2f6cbb0b57'
manifest = json.loads(matches[0].read_text())
for name, expected in manifest['files'].items():
    assert hashlib.sha256((input_dir/name).read_bytes()).hexdigest() == expected, name
sys.path.insert(0, str(input_dir))
from types import SimpleNamespace
from train_modernbert_kaggle import run
result = run(SimpleNamespace(input=input_dir, output=Path('/kaggle/working/results'), seed=20261005, family='rubert', smoke=False))


## Результаты и дальнейшая проверка

Выбранные и последние веса сохраняются отдельно. Журналы фиксируют реальные обновления, состав clean/STT и каждую проверку NLL. Больший training progress сам по себе не означает лучшие веса. Локально проверим все SHA, GPU/CPU parity, 10 сценариев и два полных замера E5 → 20 кандидатов → ID.

In [ ]:
print({k:result[k] for k in ('steps','best_step','stop_reason','elapsed_seconds')})
for profile in result['best']['metrics']:
    before = result['baseline']['metrics'][profile]['accuracy']
    after = result['best']['metrics'][profile]['accuracy']
    print(f'{profile}: {before:.1%} → {after:.1%}')
